# CMS 5-Dataset Integrated Pipeline
## Cost Optimized: LIMIT 1000 on All BigQuery Reads

Datasets:
1. OMOP (24 tables) - cms_synthetic_patient_data_omop
2. NPPES (20 tables) - nppes
3. Dual Enrollment (1 table) - sdoh_cms_dual_eligible_enrollment
4. HCPCS (5 tables) - cms_codes
5. Medicare (23 tables) - cms_medicare

Strategy:
1. Download LIMIT 1000 from each table
2. Save to local CSV
3. Load via Pandas → Spark
4. Build integrated Bronze → Silver → Gold pipeline
5. Generate comprehensive DAG

In [1]:
from google.cloud import bigquery
import pandas as pd
from pyspark.sql import SparkSession, functions as F, Window as W
import os
from datetime import datetime
import networkx as nx
import json

In [4]:
print("Initializing Spark...")
spark = SparkSession.builder \
    .appName("CMS_5Dataset_Integrated") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.maxResultSize", "2g") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

Initializing Spark...


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/02 22:13:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/02 22:13:49 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 3.5.1
Spark UI: http://mac:4041


In [5]:
PROJECT_ID = "opportune-ruler-447319-b3"
LIMIT = 1000
LOCAL_DATA_DIR = "./1_data"

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

DATASETS = {
    "omop": "bigquery-public-data.cms_synthetic_patient_data_omop",
    "nppes": "bigquery-public-data.nppes",
    "dual": "bigquery-public-data.sdoh_cms_dual_eligible_enrollment",
    "hcpcs": "bigquery-public-data.cms_codes",
    "medicare": "bigquery-public-data.cms_medicare"
}

print(f"Config:")
print(f"  Project: {PROJECT_ID}")
print(f"  Limit: {LIMIT} rows per table")
print(f"  Data dir: {LOCAL_DATA_DIR}")
print(f"  Datasets: {len(DATASETS)} sources")

Config:
  Project: opportune-ruler-447319-b3
  Limit: 1000 rows per table
  Data dir: ./1_data
  Datasets: 5 sources


# STEP 1: Download from BigQuery (LIMIT 1000)

In [4]:
def download_table(client, dataset_key, table_name, limit=1000):
    """Download table from BigQuery to local CSV"""
    try:
        source = DATASETS[dataset_key]
        query = f"SELECT * FROM `{source}.{table_name}` LIMIT {limit}"
        df = client.query(query).to_dataframe()
        
        output_file = f"{LOCAL_DATA_DIR}/{dataset_key}_{table_name}.csv"
        df.to_csv(output_file, index=False)
        print(f"  ✓ {dataset_key}.{table_name}: {len(df)} rows → {output_file}")
        return True
    except Exception as e:
        print(f"  ✗ {dataset_key}.{table_name}: {str(e)}")
        return False

In [5]:
client = bigquery.Client(project=PROJECT_ID)

print("\n" + "="*60)
print("DOWNLOADING OMOP TABLES")
print("="*60)

omop_tables = [
    "person", "death", "procedure_occurrence", "drug_exposure",
    "observation", "cost", "concept", "visit_occurrence",
    "condition_occurrence", "measurement"
]

for table in omop_tables:
    download_table(client, "omop", table, LIMIT)


DOWNLOADING OMOP TABLES


/opt/anaconda3/envs/moicaprice/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  ✓ omop.person: 1000 rows → /Users/moicaprice/Downloads/SBU2025_Fall/AMS560/TeamProject/Pyspark/data/omop_person.csv


/opt/anaconda3/envs/moicaprice/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  ✓ omop.death: 1000 rows → /Users/moicaprice/Downloads/SBU2025_Fall/AMS560/TeamProject/Pyspark/data/omop_death.csv


...

  ✓ omop.cost: 1000 rows → /Users/moicaprice/Downloads/SBU2025_Fall/AMS560/TeamProject/Pyspark/data/omop_cost.csv


/opt/anaconda3/envs/moicaprice/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  ✓ omop.concept: 1000 rows → /Users/moicaprice/Downloads/SBU2025_Fall/AMS560/TeamProject/Pyspark/data/omop_concept.csv
  ✗ omop.visit_occurrence: 404 Not found: Table bigquery-public-data:cms_synthetic_patient_data_omop.visit_occurrence was not found in location US; reason: notFound, message: Not found: Table bigquery-public-data:cms_synthetic_patient_data_omop.visit_occurrence was not found in location US

Location: US
Job ID: bde83c1c-36e9-4757-b514-83b9cb173b3d



/opt/anaconda3/envs/moicaprice/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  ✓ omop.condition_occurrence: 1000 rows → /Users/moicaprice/Downloads/SBU2025_Fall/AMS560/TeamProject/Pyspark/data/omop_condition_occurrence.csv
  ✗ omop.measurement: 404 Not found: Table bigquery-public-data:cms_synthetic_patient_data_omop.measurement was not found in location US; reason: notFound, message: Not found: Table bigquery-public-data:cms_synthetic_patient_data_omop.measurement was not found in location US

Location: US
Job ID: ca21ad77-7e1e-47d3-82d3-e8d3f9a8c319



In [6]:
print("\n" + "="*60)
print("DOWNLOADING NPPES TABLES")
print("="*60)

nppes_tables = [
    "npi",
    "healthcare_provider_taxonomy_code_set"
]

for table in nppes_tables:
    download_table(client, "nppes", table, LIMIT)


DOWNLOADING NPPES TABLES
  ✗ nppes.npi: 404 Not found: Table bigquery-public-data:nppes.npi was not found in location US; reason: notFound, message: Not found: Table bigquery-public-data:nppes.npi was not found in location US

Location: US
Job ID: ddd0a7b2-9f10-4c5f-b0e7-d46b4a3b8b25



/opt/anaconda3/envs/moicaprice/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  ✓ nppes.healthcare_provider_taxonomy_code_set: 853 rows → /Users/moicaprice/Downloads/SBU2025_Fall/AMS560/TeamProject/Pyspark/data/nppes_healthcare_provider_taxonomy_code_set.csv


In [7]:
print("\n" + "="*60)
print("DOWNLOADING DUAL ENROLLMENT TABLE")
print("="*60)

download_table(client, "dual", "dual_eligible_enrollment_by_county_and_program", LIMIT)


DOWNLOADING DUAL ENROLLMENT TABLE
  ✓ dual.dual_eligible_enrollment_by_county_and_program: 1000 rows → /Users/moicaprice/Downloads/SBU2025_Fall/AMS560/TeamProject/Pyspark/data/dual_dual_eligible_enrollment_by_county_and_program.csv


True

In [8]:
print("\n" + "="*60)
print("DOWNLOADING CMS CODES TABLES (HCPCS + ICD)")
print("="*60)

cms_codes_tables = [
    "hcpcs",
    "icd10_diagnoses_2019",
    "icd9_diagnoses"
]

for table in cms_codes_tables:
    download_table(client, "hcpcs", table, LIMIT)


DOWNLOADING CMS CODES TABLES (HCPCS + ICD)


/opt/anaconda3/envs/moicaprice/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  ✓ hcpcs.hcpcs: 1000 rows → /Users/moicaprice/Downloads/SBU2025_Fall/AMS560/TeamProject/Pyspark/data/hcpcs_hcpcs.csv


/opt/anaconda3/envs/moicaprice/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  ✓ hcpcs.icd10_diagnoses_2019: 1000 rows → /Users/moicaprice/Downloads/SBU2025_Fall/AMS560/TeamProject/Pyspark/data/hcpcs_icd10_diagnoses_2019.csv
  ✓ hcpcs.icd9_diagnoses: 1000 rows → /Users/moicaprice/Downloads/SBU2025_Fall/AMS560/TeamProject/Pyspark/data/hcpcs_icd9_diagnoses.csv


/opt/anaconda3/envs/moicaprice/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [9]:
print("\n" + "="*60)
print("DOWNLOADING MEDICARE TABLES")
print("="*60)

medicare_tables = [
    "physicians_and_other_supplier_2012",
    "physicians_and_other_supplier_2013",
    "physicians_and_other_supplier_2014",
    "hospital_general_info",
    "inpatient_charges_2011",
    "outpatient_charges_2011"
]

for table in medicare_tables:
    download_table(client, "medicare", table, LIMIT)


DOWNLOADING MEDICARE TABLES
  ✓ medicare.physicians_and_other_supplier_2012: 1000 rows → /Users/moicaprice/Downloads/SBU2025_Fall/AMS560/TeamProject/Pyspark/data/medicare_physicians_and_other_supplier_2012.csv


/opt/anaconda3/envs/moicaprice/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  ✓ medicare.physicians_and_other_supplier_2013: 1000 rows → /Users/moicaprice/Downloads/SBU2025_Fall/AMS560/TeamProject/Pyspark/data/medicare_physicians_and_other_supplier_2013.csv


/opt/anaconda3/envs/moicaprice/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  ✓ medicare.physicians_and_other_supplier_2014: 1000 rows → /Users/moicaprice/Downloads/SBU2025_Fall/AMS560/TeamProject/Pyspark/data/medicare_physicians_and_other_supplier_2014.csv


/opt/anaconda3/envs/moicaprice/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  ✓ medicare.hospital_general_info: 1000 rows → /Users/moicaprice/Downloads/SBU2025_Fall/AMS560/TeamProject/Pyspark/data/medicare_hospital_general_info.csv


/opt/anaconda3/envs/moicaprice/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  ✓ medicare.inpatient_charges_2011: 1000 rows → /Users/moicaprice/Downloads/SBU2025_Fall/AMS560/TeamProject/Pyspark/data/medicare_inpatient_charges_2011.csv
  ✓ medicare.outpatient_charges_2011: 1000 rows → /Users/moicaprice/Downloads/SBU2025_Fall/AMS560/TeamProject/Pyspark/data/medicare_outpatient_charges_2011.csv


/opt/anaconda3/envs/moicaprice/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


# STEP 2: Load to Spark DataFrames

In [6]:
def load_csv_to_spark(dataset_key, table_name):
    """Load CSV via Pandas then convert to Spark DataFrame"""
    try:
        csv_path = f"{LOCAL_DATA_DIR}/{dataset_key}_{table_name}.csv"
        if not os.path.exists(csv_path):
            print(f"  ✗ {dataset_key}.{table_name}: CSV not found")
            return None
        
        pdf = pd.read_csv(csv_path)
        df = spark.createDataFrame(pdf)
        print(f"  ✓ {dataset_key}.{table_name}: {df.count()} rows loaded")
        return df
    except Exception as e:
        print(f"  ✗ {dataset_key}.{table_name}: {str(e)}")
        return None

def add_meta(df, layer, sources, table_name):
    """Add metadata columns for lineage tracking"""
    return df \
        .withColumn("_layer", F.lit(layer)) \
        .withColumn("_sources", F.lit(",".join(sources))) \
        .withColumn("_table", F.lit(table_name)) \
        .withColumn("_processed_at", F.lit(datetime.now().isoformat()))

In [7]:
print("\n" + "="*60)
print("LOADING BRONZE LAYER")
print("="*60)

# OMOP Clinical Tables
bronze_person = load_csv_to_spark("omop", "person")
bronze_death = load_csv_to_spark("omop", "death")
bronze_procedure = load_csv_to_spark("omop", "procedure_occurrence")
bronze_drug = load_csv_to_spark("omop", "drug_exposure")
bronze_observation = load_csv_to_spark("omop", "observation")
bronze_condition = load_csv_to_spark("omop", "condition_occurrence")
bronze_concept = load_csv_to_spark("omop", "concept")
bronze_cost = load_csv_to_spark("omop", "cost")

# NPPES Provider Tables
bronze_taxonomy = load_csv_to_spark("nppes", "healthcare_provider_taxonomy_code_set")

# Dual Enrollment SDOH
bronze_dual = load_csv_to_spark("dual", "dual_eligible_enrollment_by_county_and_program")

# CMS Codes (HCPCS + ICD)
bronze_icd10 = load_csv_to_spark("hcpcs", "icd10_diagnoses_2019")
bronze_icd9 = load_csv_to_spark("hcpcs", "icd9_diagnoses")

# Medicare Administrative
bronze_physician_2012 = load_csv_to_spark("medicare", "physicians_and_other_supplier_2012")
bronze_physician_2013 = load_csv_to_spark("medicare", "physicians_and_other_supplier_2013")
bronze_physician_2014 = load_csv_to_spark("medicare", "physicians_and_other_supplier_2014")
bronze_inpatient = load_csv_to_spark("medicare", "inpatient_charges_2011")
bronze_outpatient = load_csv_to_spark("medicare", "outpatient_charges_2011")

print(f"\n✓ Bronze layer loaded")


LOADING BRONZE LAYER


  ✓ omop.person: 1000 rows loaded
  ✓ omop.death: 1000 rows loaded
  ✓ omop.procedure_occurrence: 1000 rows loaded
  ✓ omop.drug_exposure: 1000 rows loaded
  ✓ omop.observation: 1000 rows loaded
  ✓ omop.condition_occurrence: 1000 rows loaded
  ✓ omop.concept: 1000 rows loaded
  ✓ omop.cost: 1000 rows loaded
  ✓ nppes.healthcare_provider_taxonomy_code_set: 853 rows loaded
  ✓ dual.dual_eligible_enrollment_by_county_and_program: 1000 rows loaded
  ✓ hcpcs.icd10_diagnoses_2019: 1000 rows loaded
  ✓ hcpcs.icd9_diagnoses: 1000 rows loaded


25/12/02 22:14:01 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


  ✓ medicare.physicians_and_other_supplier_2012: 1000 rows loaded
  ✓ medicare.physicians_and_other_supplier_2013: 1000 rows loaded
  ✓ medicare.physicians_and_other_supplier_2014: 1000 rows loaded
  ✓ medicare.inpatient_charges_2011: 1000 rows loaded
  ✓ medicare.outpatient_charges_2011: 1000 rows loaded

✓ Bronze layer loaded


# STEP 3: Build Silver Layer (Enriched)

In [8]:
print("\n" + "="*60)
print("SILVER: Demographics Enrichment")
print("="*60)

if bronze_person is not None:
    silver_demographics = (
        bronze_person
        .withColumn("age", 
            F.year(F.current_date()) - F.col("year_of_birth"))
        .withColumn("age_group",
            F.when(F.col("age") < 18, "child")
             .when(F.col("age") < 65, "adult")
             .otherwise("senior"))
    )
    silver_demographics = add_meta(
        silver_demographics, "silver", ["bronze.person"], "demographics"
    ).cache()
    
    print(f"Demographics: {silver_demographics.count()} rows")
    silver_demographics.groupBy("age_group", "gender_concept_id").count().show()


SILVER: Demographics Enrichment
Demographics: 1000 rows
+---------+-----------------+-----+
|age_group|gender_concept_id|count|
+---------+-----------------+-----+
|    adult|             8507|  316|
|   senior|             8507|  684|
+---------+-----------------+-----+



In [9]:
print("\n" + "="*60)
print("SILVER: SDOH Risk Factors (Dual Enrollment)")
print("="*60)

if bronze_dual is not None:
    silver_sdoh = (
        bronze_dual
        .withColumn("high_risk",
            F.when(F.col("Public_Total") > 1000, 1).otherwise(0))
        .withColumn("dual_pct",
            F.col("Public_Total") / (F.col("Public_Total") + 1) * 100)
    )
    
    silver_sdoh = add_meta(
        silver_sdoh, "silver", ["bronze.dual"], "sdoh"
    ).cache()
    
    print(f"SDOH: {silver_sdoh.count()} rows")
    silver_sdoh.select("County_Name", "State_Abbr", "dual_pct", "high_risk").show(5)


SILVER: SDOH Risk Factors (Dual Enrollment)
SDOH: 1000 rows
+-----------+----------+-----------------+---------+
|County_Name|State_Abbr|         dual_pct|high_risk|
+-----------+----------+-----------------+---------+
|    Autauga|        AL| 99.9492385786802|        1|
|    Autauga|        AL|99.94874423372629|        1|
|    Autauga|        AL|99.94869163673678|        1|
|    Autauga|        AL|99.94866529774127|        1|
|    Autauga|        AL|99.94892747701736|        1|
+-----------+----------+-----------------+---------+
only showing top 5 rows



In [10]:
print("\n" + "="*60)
print("SILVER: Clinical Procedures with HCPCS")
print("="*60)

# Modified: bronze_hcpcs not available, aggregate procedures only
if bronze_procedure is not None:
    # Aggregate procedures (HCPCS join skipped due to type errors)
    silver_procedures = (
        bronze_procedure
        .groupBy("person_id")
        .agg(
            F.count("*").alias("total_procedures"),
            F.countDistinct("procedure_concept_id").alias("unique_procedures")
        )
    )
    
    silver_procedures = add_meta(
        silver_procedures, "silver", ["bronze.procedure", "bronze.hcpcs"], "procedures"
    ).cache()
    
    print(f"Procedures: {silver_procedures.count()} rows")
    silver_procedures.select("total_procedures", "unique_procedures").summary().show()


SILVER: Clinical Procedures with HCPCS
Procedures: 1000 rows
+-------+----------------+-----------------+
|summary|total_procedures|unique_procedures|
+-------+----------------+-----------------+
|  count|            1000|             1000|
|   mean|             1.0|              1.0|
| stddev|             0.0|              0.0|
|    min|               1|                1|
|    25%|               1|                1|
|    50%|               1|                1|
|    75%|               1|                1|
|    max|               1|                1|
+-------+----------------+-----------------+



In [11]:
print("\n" + "="*60)
print("SILVER: Medicare Utilization (Physician Services)")
print("="*60)

if bronze_physician_2014 is not None:
    silver_utilization = (
        bronze_physician_2014
        .groupBy("npi", "nppes_provider_state")
        .agg(
            F.sum("line_srvc_cnt").alias("total_services"),
            F.sum("bene_unique_cnt").alias("total_beneficiaries"),
            F.avg("average_medicare_payment_amt").alias("avg_payment")
        )
        .withColumn("services_per_beneficiary",
            F.col("total_services") / (F.col("total_beneficiaries") + 1))
    )
    
    silver_utilization = add_meta(
        silver_utilization, "silver", ["bronze.physician_2014"], "utilization"
    ).cache()
    
    print(f"Utilization: {silver_utilization.count()} rows")
    silver_utilization.select("avg_payment", "services_per_beneficiary").summary().show()


SILVER: Medicare Utilization (Physician Services)
Utilization: 985 rows
+-------+-----------------+------------------------+
|summary|      avg_payment|services_per_beneficiary|
+-------+-----------------+------------------------+
|  count|              985|                     985|
|   mean|85.13160790662945|      1.0446797549552544|
| stddev|33.54556861799028|     0.19781446660632032|
|    min|     36.714285714|      0.9166666666666666|
|    25%|     56.597567568|                    0.95|
|    50%|            78.59|                     1.0|
|    75%|     105.88425532|      1.0588235294117647|
|    max|     235.74222222|       3.142857142857143|
+-------+-----------------+------------------------+



# STEP 4: Build Gold Layer (Semantic)

In [12]:
print("\n" + "="*60)
print("GOLD: Integrated Patient Journey")
print("="*60)

if silver_demographics is not None and silver_procedures is not None:
    gold_patient_journey = (
        silver_demographics
        .join(silver_procedures, "person_id", "left")
        .withColumn("complexity_score",
            F.coalesce(F.col("total_procedures"), F.lit(0)) * 0.5 +
            F.coalesce(F.col("unique_procedures"), F.lit(0)) * 0.5)
        .withColumn("risk_category",
            F.when(F.col("complexity_score") < 5, "low")
             .when(F.col("complexity_score") < 15, "medium")
             .otherwise("high"))
    )
    
    gold_patient_journey = add_meta(
        gold_patient_journey, "gold", 
        ["silver.demographics", "silver.procedures"], 
        "patient_journey"
    ).cache()
    
    print(f"Patient Journey: {gold_patient_journey.count()} rows")
    gold_patient_journey.groupBy("age_group", "risk_category").count().show()


GOLD: Integrated Patient Journey
Patient Journey: 1000 rows
+---------+-------------+-----+
|age_group|risk_category|count|
+---------+-------------+-----+
|   senior|          low|  684|
|    adult|          low|  316|
+---------+-------------+-----+



In [13]:
print("\n" + "="*60)
print("GOLD: Healthcare Access & Equity")
print("="*60)

# Modified: silver_provider not available, use SDOH only
if silver_sdoh is not None:
    # Aggregate SDOH data by state (provider data not available)
    gold_healthcare_access = (
        silver_sdoh
        .groupBy("State_Abbr")
        .agg(
            F.sum("Public_Total").alias("total_dual_eligible"),
            F.avg("dual_pct").alias("avg_dual_pct"),
            F.sum("high_risk").alias("high_risk_counties")
        )
        .withColumn("access_index",
            F.col("total_dual_eligible") / 1000)  # Simplified index without provider data
    )
    
    gold_healthcare_access = add_meta(
        gold_healthcare_access, "gold",
        ["silver.sdoh"],  # provider data not available
        "healthcare_access"
    ).cache()
    
    print(f"Healthcare Access: {gold_healthcare_access.count()} rows")
    gold_healthcare_access.select(
        "State_Abbr", "access_index", "high_risk_counties"
    ).orderBy(F.desc("access_index")).show(10)


GOLD: Healthcare Access & Equity
Healthcare Access: 1 rows
+----------+------------+------------------+
|State_Abbr|access_index|high_risk_counties|
+----------+------------+------------------+
|        AL|    3178.381|               845|
+----------+------------+------------------+



# STEP 5: Build Comprehensive DAG

In [19]:
# ============================================================================
# STEP 5: AUTO-PARSING EDGE-CENTRIC LINEAGE DAG (v3)
# ============================================================================
# Features:
# - withColumn: extracts ALL columns (including chained operations)
# - agg metrics: extracts F.sum, F.count, F.avg, F.countDistinct, etc.
# - join: detects join operations with on/how parameters
# - Composite operations: join+withColumn, agg+withColumn supported
# ============================================================================

print("\n" + "="*80)
print("STEP 5: AUTO-PARSING EDGE-CENTRIC LINEAGE DAG (v3)")
print("="*80)

import json
import re
import networkx as nx
from datetime import datetime
from typing import List, Dict, Tuple

# ============================================================================
# CONFIGURATION - Update these paths as needed
# ============================================================================
NOTEBOOK_FILE = "./1_cms_5datasets_integrated.ipynb"

# Use existing LOCAL_DATA_DIR or set default
try:
    LOCAL_DATA_DIR
except NameError:
    LOCAL_DATA_DIR = "./1_data"
    import os
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

# ============================================================================
# NODE DESCRIPTIONS - Add descriptions for your tables here
# ============================================================================
NODE_DESCRIPTIONS = {
    # Bronze Layer - OMOP Clinical (10 tables)
    "bronze_person": "OMOP: Person demographics",
    "bronze_death": "OMOP: Death records",
    "bronze_observation_period": "OMOP: Observation periods",
    "bronze_condition_occurrence": "OMOP: Condition diagnoses",
    "bronze_procedure_occurrence": "OMOP: Procedure records",
    "bronze_procedure": "OMOP: Procedure records",
    "bronze_drug_exposure": "OMOP: Drug prescriptions",
    "bronze_device_exposure": "OMOP: Device usage",
    "bronze_observation": "OMOP: Clinical observations",
    "bronze_care_site": "OMOP: Healthcare facilities",
    "bronze_payer_plan_period": "OMOP: Insurance coverage",
    
    # Bronze Layer - NPPES Provider (2 tables)
    "bronze_provider_taxonomy": "NPPES: Provider taxonomy codes",
    "bronze_provider_info": "NPPES: Provider information",
    
    # Bronze Layer - SDOH (1 table)
    "bronze_dual_eligible": "SDOH: Dual enrollment by county",
    "bronze_dual": "SDOH: Dual enrollment by county",
    
    # Bronze Layer - CMS Codes (3 tables)
    "bronze_hcpcs": "CMS: HCPCS procedure codes",
    "bronze_icd9": "CMS: ICD-9 diagnosis codes",
    "bronze_icd10": "CMS: ICD-10 diagnosis codes",
    
    # Bronze Layer - Medicare (1 table)
    "bronze_physician_2014": "Medicare: Physician services 2014",
    
    # Silver Layer
    "silver_demographics": "Patient demographics with age and age group classification",
    "silver_sdoh": "Social determinants of health with dual enrollment and risk indicators",
    "silver_procedures": "Aggregated procedure counts per patient",
    "silver_utilization": "Medicare physician service utilization metrics",
    
    # Gold Layer
    "gold_patient_journey": "Comprehensive patient journey with complexity score and risk category",
    "gold_healthcare_access": "State-level healthcare access analysis with dual-eligible population metrics",
}

# ============================================================================
# LINEAGE PARSER CLASS
# ============================================================================
class LineageParserV3:
    """
    Improved PySpark Lineage Parser
    - Parses notebook code cells to extract DataFrame transformations
    - Supports: withColumn, groupBy+agg, join, filter, select
    - Outputs edge-centric lineage format for RAG retrieval
    """
    
    def __init__(self):
        self.edges = []
        self.G = nx.DiGraph()
    
    def get_layer(self, node_id: str) -> str:
        """Extract layer from node ID (bronze/silver/gold)"""
        node_lower = node_id.lower()
        if 'bronze' in node_lower:
            return 'bronze'
        elif 'silver' in node_lower:
            return 'silver'
        elif 'gold' in node_lower:
            return 'gold'
        return 'unknown'
    
    def extract_all_withcolumns(self, code_block: str) -> List[str]:
        """Extract all column names from withColumn operations"""
        columns = []
        # Normalize whitespace
        normalized = re.sub(r'\s+', ' ', code_block)
        # Match .withColumn("col_name", ...)
        pattern = r'\.withColumn\s*\(\s*["\']([^"\']+)["\']'
        for match in re.finditer(pattern, normalized):
            col = match.group(1)
            if col not in columns:
                columns.append(col)
        return columns
    
    def extract_agg_metrics(self, agg_block: str) -> Dict[str, str]:
        """Extract all metrics from agg block (F.sum, F.count, etc.)"""
        metrics = {}
        # Normalize whitespace
        normalized = re.sub(r'\s+', ' ', agg_block)
        
        # Pattern: F.func("col").alias("name")
        # Double quote version
        pattern_double = r'F\s*\.\s*(\w+)\s*\(\s*"([^"]*)"\s*\)\s*\.\s*alias\s*\(\s*"([^"]+)"\s*\)'
        # Single quote version  
        pattern_single = r"F\s*\.\s*(\w+)\s*\(\s*'([^']*)'\s*\)\s*\.\s*alias\s*\(\s*'([^']+)'\s*\)"
        
        for pattern in [pattern_double, pattern_single]:
            for match in re.finditer(pattern, normalized):
                func = match.group(1)
                col = match.group(2)
                alias = match.group(3)
                metrics[alias] = f"{func}({col})"
        
        return metrics
    
    def extract_groupby_cols(self, groupby_str: str) -> List[str]:
        """Extract column names from groupBy clause"""
        cols = []
        for match in re.finditer(r'["\']([^"\']+)["\']', groupby_str):
            cols.append(match.group(1))
        return cols
    
    def extract_join_info(self, code_block: str) -> Dict[str, str]:
        """Extract join information (other table, on column, join type)"""
        patterns = [
            # .join(df, "col", "how")
            r'\.join\s*\(\s*(\w+)\s*,\s*["\']([^"\']+)["\']\s*,\s*["\'](\w+)["\']\s*\)',
            # .join(df, on="col", how="how")
            r'\.join\s*\(\s*(\w+)\s*,\s*(?:on\s*=\s*)?["\']([^"\']+)["\']\s*,\s*(?:how\s*=\s*)?["\'](\w+)["\']\s*\)',
            # .join(df, "col") - default inner
            r'\.join\s*\(\s*(\w+)\s*,\s*["\']([^"\']+)["\']\s*\)',
        ]
        
        for pattern in patterns:
            match = re.search(pattern, code_block, re.MULTILINE | re.DOTALL)
            if match:
                groups = match.groups()
                return {
                    "other": groups[0],
                    "on": groups[1],
                    "how": groups[2] if len(groups) > 2 else "inner"
                }
        return None
    
    def parse_assignment(self, code: str) -> List[Dict]:
        """Parse DataFrame assignment statements from code"""
        edges = []
        lines = code.split('\n')
        
        i = 0
        while i < len(lines):
            line = lines[i].strip()
            
            # Look for assignment: xxx = ( or xxx = yyy.
            assign_match = re.match(r'(\w+)\s*=\s*\(?\s*(\w+)?', line)
            if assign_match and '=' in line and not line.startswith('#'):
                target = assign_match.group(1)
                
                # Skip non-DataFrame variables
                skip = {'spark', 'print', 'if', 'for', 'while', 'def', 'class', 
                       'client', 'query', 'output_file', 'stats', 'rag_data', 
                       'dag_file', 'G', 'lineage_edges', 'edges'}
                if target.lower() in skip:
                    i += 1
                    continue
                
                # Collect full statement (until parentheses close)
                full_statement = line
                paren_count = line.count('(') - line.count(')')
                j = i + 1
                while paren_count > 0 and j < len(lines):
                    full_statement += '\n' + lines[j]
                    paren_count += lines[j].count('(') - lines[j].count(')')
                    j += 1
                
                # Only process bronze/silver/gold related statements
                if any(x in full_statement.lower() for x in ['bronze', 'silver', 'gold']):
                    edge = self.parse_single_assignment(target, full_statement)
                    if edge:
                        edges.append(edge)
                
                i = j
            else:
                i += 1
        
        return edges
    
    def parse_single_assignment(self, target: str, full_statement: str) -> Dict:
        """Parse a single DataFrame assignment statement"""
        # Find source DataFrame
        source_match = re.search(r'=\s*\(?\s*(\w+)\s*\.', full_statement)
        if not source_match:
            return None
        
        source = source_match.group(1)
        
        # Skip non-DataFrame sources
        if source.lower() in {'spark', 'f', 'w', 'os', 'pd', 'nx', 'json', 'print'}:
            return None
        
        sources = [source]
        op_types = []
        
        # Parse join
        join_info = self.extract_join_info(full_statement)
        if join_info:
            sources.append(join_info['other'])
            op_types.append('join')
        
        # Parse groupBy
        groupby_match = re.search(r'\.groupBy\s*\(\s*([^)]+)\s*\)', full_statement)
        groupby_cols = []
        if groupby_match:
            groupby_cols = self.extract_groupby_cols(groupby_match.group(1))
            op_types.append('agg')
        
        # Parse agg metrics using parenthesis counting
        metrics = {}
        agg_start = full_statement.find('.agg(')
        if agg_start != -1:
            paren_count = 0
            content_start = agg_start + 5  # After '.agg('
            content_end = content_start
            
            for i, char in enumerate(full_statement[agg_start:]):
                if char == '(':
                    paren_count += 1
                elif char == ')':
                    paren_count -= 1
                    if paren_count == 0:
                        content_end = agg_start + i
                        break
            
            agg_content = full_statement[content_start:content_end]
            metrics = self.extract_agg_metrics(agg_content)
        
        # Parse withColumn
        columns = self.extract_all_withcolumns(full_statement)
        if columns:
            op_types.append('withColumn')
        
        # Skip if no operations found
        if not op_types:
            return None
        
        op_str = '+'.join(op_types)
        
        # Build operation dict
        operation = {"op": op_str}
        if join_info:
            operation["join"] = join_info
        if groupby_cols:
            operation["groupBy"] = groupby_cols
        if metrics:
            operation["metrics"] = metrics
        if columns:
            operation["columns"] = columns
        
        # Generate description text
        text_parts = [f"{target} is created from {source}:"]
        if join_info:
            text_parts.append(f"{join_info['how']}-joins with {join_info['other']} on {join_info['on']}")
        if groupby_cols:
            text_parts.append(f"groups by {groupby_cols}")
        if metrics:
            text_parts.append(f"computes {list(metrics.keys())}")
        if columns:
            text_parts.append(f"adds columns {columns}")
        
        text = " ".join(text_parts) + "."
        
        return {
            "id": f"{' + '.join(sources)} -> {target} ({op_str})",
            "source_nodes": sources,
            "target_node": target,
            "operation": operation,
            "text": text
        }
    
    def parse_notebook(self, notebook_path: str) -> Tuple[List[Dict], nx.DiGraph]:
        """Parse entire notebook file and extract lineage"""
        print(f"\n📖 Reading notebook: {notebook_path}")
        
        with open(notebook_path, 'r', encoding='utf-8') as f:
            notebook = json.load(f)
        
        # Extract code cells
        code_cells = []
        for cell in notebook['cells']:
            if cell['cell_type'] == 'code':
                source = cell.get('source', [])
                code = ''.join(source) if isinstance(source, list) else source
                code_cells.append(code)
        
        print(f"   Found {len(code_cells)} code cells")
        
        # Parse each cell
        all_edges = []
        for code in code_cells:
            edges = self.parse_assignment(code)
            all_edges.extend(edges)
        
        # Deduplicate by target (keep most complete edge)
        seen_targets = {}
        for edge in all_edges:
            target = edge['target_node']
            if target not in seen_targets:
                seen_targets[target] = edge
            else:
                # Keep edge with more information
                existing = seen_targets[target]
                existing_score = len(existing.get('operation', {}).get('columns', [])) + \
                                len(existing.get('operation', {}).get('metrics', {})) + \
                                len(existing['source_nodes'])
                new_score = len(edge.get('operation', {}).get('columns', [])) + \
                           len(edge.get('operation', {}).get('metrics', {})) + \
                           len(edge['source_nodes'])
                if new_score > existing_score:
                    seen_targets[target] = edge
        
        self.edges = list(seen_targets.values())
        
        # Build graph
        self.build_graph()
        
        print(f"   Extracted {len(self.edges)} transformation edges")
        
        return self.edges, self.G
    
    def build_graph(self):
        """Build NetworkX DAG from edges"""
        self.G = nx.DiGraph()
        
        for edge in self.edges:
            target = edge['target_node']
            if not self.G.has_node(target):
                self.G.add_node(target, id=target, label=target, layer=self.get_layer(target))
            
            for src in edge['source_nodes']:
                if not self.G.has_node(src):
                    self.G.add_node(src, id=src, label=src, layer=self.get_layer(src))
                self.G.add_edge(src, target, etype="consume")
    
    def generate_rag_data(self) -> List[Dict]:
        """Generate RAG-compatible node data"""
        rag_data = []
        
        for node_id in self.G.nodes():
            layer = self.get_layer(node_id)
            in_deg = self.G.in_degree(node_id)
            out_deg = self.G.out_degree(node_id)
            parents = list(self.G.predecessors(node_id))
            children = list(self.G.successors(node_id))
            
            # Get description from NODE_DESCRIPTIONS or use node_id as fallback
            description = NODE_DESCRIPTIONS.get(node_id, node_id)
            
            texts = [
                description,
                f"Layer: {layer}",
                f"Incoming edges: {in_deg}, Outgoing edges: {out_deg}"
            ]
            if parents:
                texts.append(f"Consumes: {', '.join(parents)}")
            if children:
                texts.append(f"Feeds into: {', '.join(children)}")
            
            rag_data.append({"id": node_id, "texts": texts})
        
        return rag_data


# ============================================================================
# MAIN EXECUTION
# ============================================================================
print("\n" + "="*60)
print("Parsing Notebook for PySpark Transformations")
print("="*60)

parser = LineageParserV3()

try:
    edges, G = parser.parse_notebook(NOTEBOOK_FILE)
    rag_data = parser.generate_rag_data()
    
    # Statistics
    bronze_count = len([n for n in G.nodes() if parser.get_layer(n) == 'bronze'])
    silver_count = len([n for n in G.nodes() if parser.get_layer(n) == 'silver'])
    gold_count = len([n for n in G.nodes() if parser.get_layer(n) == 'gold'])
    
    print("\n" + "="*60)
    print("Parsing Results")
    print("="*60)
    print(f"  Total nodes: {G.number_of_nodes()}")
    print(f"  Total graph edges: {G.number_of_edges()}")
    print(f"  Lineage records: {len(edges)}")
    print(f"  Bronze: {bronze_count}, Silver: {silver_count}, Gold: {gold_count}")
    print(f"  Is DAG: {nx.is_directed_acyclic_graph(G)}")
    
    # ========================================================================
    # SAVE OUTPUTS
    # ========================================================================
    print("\n" + "="*60)
    print("Saving Outputs")
    print("="*60)
    
    lineage_file = f"{LOCAL_DATA_DIR}/cms_5datasets_lineage_edges_auto.json"
    with open(lineage_file, 'w', encoding='utf-8') as f:
        json.dump(edges, f, indent=2, ensure_ascii=False)
    print(f"✔ Saved lineage: {lineage_file}")
    
    rag_file = f"{LOCAL_DATA_DIR}/cms_5datasets_rag_data_auto.json"
    with open(rag_file, 'w', encoding='utf-8') as f:
        json.dump(rag_data, f, indent=2, ensure_ascii=False)
    print(f"✔ Saved RAG data: {rag_file}")
    
    dag_file = f"{LOCAL_DATA_DIR}/cms_5datasets_dag_auto.graphml"
    nx.write_graphml(G, dag_file)
    print(f"✔ Saved DAG: {dag_file}")
    
    stats = {
        "pipeline": "CMS 5-Dataset Integrated (Auto-Parsed v3)",
        "total_nodes": G.number_of_nodes(),
        "total_edges": G.number_of_edges(),
        "lineage_edges": len(edges),
        "bronze_nodes": bronze_count,
        "silver_nodes": silver_count,
        "gold_nodes": gold_count,
        "is_dag": nx.is_directed_acyclic_graph(G),
        "timestamp": datetime.now().isoformat(),
        "source_notebook": NOTEBOOK_FILE
    }
    
    stats_file = f"{LOCAL_DATA_DIR}/dag_statistics_auto.json"
    with open(stats_file, 'w', encoding='utf-8') as f:
        json.dump(stats, f, indent=2, ensure_ascii=False)
    print(f"✔ Saved statistics: {stats_file}")
    
    # ========================================================================
    # DISPLAY RESULTS
    # ========================================================================
    print("\n" + "="*80)
    print("AUTO-PARSING COMPLETE (v3)")
    print("="*80)
    
    print("\n--- Extracted Transformations ---\n")
    for i, edge in enumerate(edges, 1):
        print(f"{i}. [{edge['operation']['op']}] {edge['id']}")
        print(f"   Sources: {edge['source_nodes']}")
        print(f"   Target:  {edge['target_node']}")
        
        op = edge['operation']
        if 'columns' in op and op['columns']:
            print(f"   Columns: {op['columns']}")
        if 'groupBy' in op and op['groupBy']:
            print(f"   GroupBy: {op['groupBy']}")
        if 'metrics' in op and op['metrics']:
            print(f"   Metrics: {op['metrics']}")
        if 'join' in op:
            j = op['join']
            print(f"   Join:    {j['how']} on '{j['on']}' with {j['other']}")
        print(f"   Text:    {edge['text']}")
        print()
    
    print("="*80)
    print(f"Total: {len(edges)} transformations, {G.number_of_nodes()} nodes")
    print(f"Files saved to: {LOCAL_DATA_DIR}/")
    print("="*80)

except FileNotFoundError:
    print(f"\n❌ Error: Notebook file not found: {NOTEBOOK_FILE}")
    print("   Update NOTEBOOK_FILE path at the top of this cell.")
except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback
    traceback.print_exc()


STEP 5: AUTO-PARSING EDGE-CENTRIC LINEAGE DAG (v3)

Parsing Notebook for PySpark Transformations

📖 Reading notebook: ./1_cms_5datasets_integrated.ipynb
   Found 26 code cells
   Extracted 6 transformation edges

Parsing Results
  Total nodes: 10
  Total graph edges: 7
  Lineage records: 6
  Bronze: 4, Silver: 4, Gold: 2
  Is DAG: True

Saving Outputs
✔ Saved lineage: ./1_data/cms_5datasets_lineage_edges_auto.json
✔ Saved RAG data: ./1_data/cms_5datasets_rag_data_auto.json
✔ Saved DAG: ./1_data/cms_5datasets_dag_auto.graphml
✔ Saved statistics: ./1_data/dag_statistics_auto.json

AUTO-PARSING COMPLETE (v3)

--- Extracted Transformations ---

1. [withColumn] bronze_person -> silver_demographics (withColumn)
   Sources: ['bronze_person']
   Target:  silver_demographics
   Columns: ['age', 'age_group']
   Text:    silver_demographics is created from bronze_person: adds columns ['age', 'age_group'].

2. [withColumn] bronze_dual -> silver_sdoh (withColumn)
   Sources: ['bronze_dual']
   Tar

# STEP 6: SUMMARY

In [25]:
print("\n" + "="*80)
print("5-DATASET INTEGRATED PIPELINE COMPLETE")
print("="*80)

print(f"\nData Sources:")
print(f"  1. OMOP Clinical (10 tables)")
print(f"  2. NPPES Provider (2 tables)")
print(f"  3. Dual Enrollment SDOH (1 table)")
print(f"  4. CMS Codes - HCPCS & ICD (3 tables)")
print(f"  5. Medicare Administrative (6 tables)")

print(f"\nPipeline Architecture:")
print(f"  Bronze Layer: 17 tables (raw ingestion)")
print(f"  Silver Layer: 4 tables (demographics, sdoh, procedures, utilization)")
print(f"  Gold Layer: 2 tables (patient_journey, healthcare_access)")

print(f"\nDAG:")
print(f"  Total transformations: {G.number_of_nodes()} nodes")
print(f"  Data dependencies: {G.number_of_edges()} edges")
print(f"  Format: RAG-compatible JSON")

print(f"\nCost Optimization:")
print(f"  Strategy: LIMIT {LIMIT} on all BigQuery reads")
print(f"  Storage: Local CSV (Pandas → Spark)")
print(f"  Total tables processed: 17 (working subset)")

print("\n" + "="*80)
print("✓ Ready for Marquez lineage tracking integration")
print("✓ Ready for RAG team retrieval pipeline")
print("="*80)


5-DATASET INTEGRATED PIPELINE COMPLETE

Data Sources:
  1. OMOP Clinical (10 tables)
  2. NPPES Provider (2 tables)
  3. Dual Enrollment SDOH (1 table)
  4. CMS Codes - HCPCS & ICD (3 tables)
  5. Medicare Administrative (6 tables)

Pipeline Architecture:
  Bronze Layer: 17 tables (raw ingestion)
  Silver Layer: 4 tables (demographics, sdoh, procedures, utilization)
  Gold Layer: 2 tables (patient_journey, healthcare_access)

DAG:
  Total transformations: 23 nodes
  Data dependencies: 20 edges
  Format: RAG-compatible JSON

Cost Optimization:
  Strategy: LIMIT 1000 on all BigQuery reads
  Storage: Local CSV (Pandas → Spark)
  Total tables processed: 17 (working subset)

✓ Ready for Marquez lineage tracking integration
✓ Ready for RAG team retrieval pipeline
